# **Computational Drug Discovery [Part 1] Download Bioactivity Data**


## **ChEMBL Database**

The [*ChEMBL Database*](https://www.ebi.ac.uk/chembl/) is a database that contains curated bioactivity data of more than 2 million compounds. It is compiled from more than 76,000 documents, 1.2 million assays and the data spans 13,000 targets and 1,800 cells and 33,000 indications.
[Data as of March 25, 2020; ChEMBL version 26].

## **Installing libraries**

Install the ChEMBL web service package so that we can retrieve bioactivity data from the ChEMBL Database.

In [21]:
! pip install chembl_webresource_client

## **Importing libraries**

In [22]:
# Import necessary libraries
import pandas as pd
from chembl_webresource_client.new_client import new_client

## **Search for Target protein**

### **Target search for keap1**

In [23]:

target = new_client.target
target_query = target.search('keap1')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Keap1/Nrf2,17.0,False,CHEMBL3038498,"[{'accession': 'Q16236', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
1,[],Rattus norvegicus,KEAP1/NRF2,17.0,False,CHEMBL4296095,"[{'accession': 'O54968', 'component_descriptio...",PROTEIN COMPLEX,10116
2,[],Rattus norvegicus,Kelch-like ECH-associated protein 1,13.0,False,CHEMBL4523596,"[{'accession': 'P57790', 'component_descriptio...",SINGLE PROTEIN,10116
3,[],Mus musculus,Kelch-like ECH-associated protein 1,12.0,False,CHEMBL3562164,"[{'accession': 'Q9Z2X8', 'component_descriptio...",SINGLE PROTEIN,10090
4,[],Homo sapiens,Kelch-like ECH-associated protein 1,11.0,False,CHEMBL2069156,"[{'accession': 'Q14145', 'component_descriptio...",SINGLE PROTEIN,9606
5,[],Mus musculus,Protein cereblon/Kelch-like ECH-associated pro...,11.0,False,CHEMBL5465249,"[{'accession': 'Q9Z2X8', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,10090
6,[],Homo sapiens,Kelch-like ECH-associated protein 1/Bromodomai...,10.0,False,CHEMBL6193832,"[{'accession': 'O60885', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
7,[],Homo sapiens,Kelch-like ECH-associated protein 1/Microtubul...,9.0,False,CHEMBL4296122,"[{'accession': 'P10636', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
8,[],Homo sapiens,Sequestosome-1/Kelch-like ECH-associated prote...,8.0,False,CHEMBL4106129,"[{'accession': 'Q14145', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606
9,[],Mus musculus,Kelch-like ECH-associated protein 1/Nuclear fa...,8.0,False,CHEMBL4296107,"[{'accession': 'Q60795', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,10090


In [24]:
selected_target = targets.target_chembl_id[4]
selected_target

'CHEMBL2069156'

In [25]:
import os
import pandas as pd
from chembl_webresource_client.new_client import new_client

OUTPUT_DIR = '' # Define OUTPUT_DIR. Set to 'data/' or similar if you want a subfolder.
RAW_CSV = OUTPUT_DIR + 'keap1_raw_chembl.csv'

if os.path.exists(RAW_CSV):
    print('Loading saved ChEMBL data...')
    df_raw = pd.read_csv(RAW_CSV)
    print(f'Loaded {len(df_raw)} records from cache.')
else:
    print('Querying ChEMBL (this may take a minute)...')
    activity = new_client.activity
    keap1_raw = list(activity.filter(
        target_chembl_id       = 'CHEMBL2069156',
        standard_type          = 'IC50',
        standard_units         = 'nM',
        standard_relation      = '=',
        assay_confidence_score = 9,
        pchembl_value__isnull  = False,
        canonical_smiles__isnull = False,
    ).only([
        'molecule_chembl_id',
        'canonical_smiles',
        'standard_value',
        'standard_units',
    ])[:10000])

    df_raw = pd.DataFrame(keap1_raw)
    df_raw['standard_value'] = pd.to_numeric(
        df_raw['standard_value'], errors='coerce')
    df_raw = df_raw.rename(columns={'standard_value': 'IC50_nM'})
    df_raw = df_raw.dropna(subset=['canonical_smiles', 'IC50_nM'])
    df_raw = df_raw[df_raw['IC50_nM'] > 0]
    df_raw.to_csv(RAW_CSV, index=False)
    print(f'Query complete. {len(df_raw)} records saved to {RAW_CSV}')

print(f'Unique compounds: {df_raw["molecule_chembl_id"].nunique()}')
df_raw.head()


Querying ChEMBL (this may take a minute)...
Query complete. 282 records saved to keap1_raw_chembl.csv
Unique compounds: 260


,canonical_smiles,molecule_chembl_id,standard_units,IC50_nM,units,value
0,COc1ccc(S(=O)(=O)Nc2ccc(NS(=O)(=O)c3ccc(OC)cc3...,CHEMBL2402207,nM,2700.0,uM,2.7
1,CC(C)C[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](...,CHEMBL2402205,nM,33.0,nM,33.0
2,COc1ccc(S(=O)(=O)Nc2ccc(NS(=O)(=O)c3ccc(OC)cc3...,CHEMBL2402207,nM,1700.0,nM,1700.0
3,COc1ccc(S(=O)(=O)N(CC(=O)O)c2ccc(N(CC(=O)O)S(=...,CHEMBL3237245,nM,20.0,nM,20.0
4,COc1cccc(S(=O)(=O)Nc2ccc(NS(=O)(=O)c3cccc(OC)c...,CHEMBL3632699,nM,2300.0,nM,2300.0


Labeling compounds as either being active, inactive or intermediate
The bioactivity data is in the IC50 unit. Compounds having values of less than 1000 nM will be considered to be active while those greater than 10,000 nM will be considered to be inactive. As for those values in between 1,000 and 10,000 nM will be referred to as intermediate.


In [26]:
import numpy as np
import pandas as pd

def geometric_mean_ic50(values):
    """Geometric mean of a list of positive IC50 values."""
    vals = [v for v in values if v > 0]
    if not vals:
        return np.nan
    return float(np.exp(np.mean(np.log(vals))))

df_dedup = (
    df_raw.groupby('molecule_chembl_id')
    .agg(
        canonical_smiles=('canonical_smiles', 'first'),
        IC50_nM_list=('IC50_nM', list),
    )
    .reset_index()
)
df_dedup['IC50_nM'] = df_dedup['IC50_nM_list'].apply(geometric_mean_ic50)
df_dedup = df_dedup.drop(columns='IC50_nM_list')
df_dedup = df_dedup.dropna(subset=['IC50_nM'])

bioactivity_class = []
for i in df_dedup.IC50_nM:
  if float(i) >= 10000:
    bioactivity_class.append("inactive")
  elif float(i) <= 1000:
    bioactivity_class.append("active")
  else:
    bioactivity_class.append("intermediate")

In [27]:
data_tuples = list(zip(df_dedup['molecule_chembl_id'], df_dedup['canonical_smiles'], bioactivity_class, df_dedup['IC50_nM']))
df = pd.DataFrame( data_tuples,  columns=['molecule_chembl_id', 'canonical_smiles', 'bioactivity_class', 'IC50_nM'])

In [28]:
df_dedup.to_csv('dedup_preprocessed_data.csv', index=False)

In [29]:
import sys
if 'rdkit' not in sys.modules:
  !pip install rdkit

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def compute_descriptors(mol):
    """Compute a standard set of physicochemical descriptors."""
    return {
        'MW': round(Descriptors.MolWt(mol), 2),
        'LogP': round(Descriptors.MolLogP(mol), 2),
        'HBD': rdMolDescriptors.CalcNumHBD(mol),
        'HBA': rdMolDescriptors.CalcNumHBA(mol),
        'RotB': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'TPSA': round(rdMolDescriptors.CalcTPSA(mol), 2),
        'Rings': rdMolDescriptors.CalcNumRings(mol),
        'ArRings': rdMolDescriptors.CalcNumAromaticRings(mol),
    }

In [30]:
import numpy as np
from rdkit import Chem

# Parse SMILES
df_dedup["mol"] = df_dedup["canonical_smiles"].apply(Chem.MolFromSmiles)
n_before = len(df_dedup)
df_dedup = df_dedup[df_dedup["mol"].notna()].copy()
print(f"Valid structures: {len(df_dedup)} / {n_before}")

Valid structures: 260 / 260


In [31]:
# Descriptors
print("Computing descriptors...")
desc_rows = [compute_descriptors(mol) for mol in df_dedup["mol"]]
desc_df = pd.DataFrame(desc_rows, index=df_dedup.index)
for col in desc_df.columns:
    df_dedup[col] = desc_df[col]

Computing descriptors...


In [32]:
# 1. Compute pIC50 safely directly on df_dedup
df_dedup["pIC50"] = 9 - np.log10(df_dedup["IC50_nM"])

# 2. Add the bioactivity classification directly
df_dedup["bioactivity_class"] = bioactivity_class

# 3. Select final columns for output
final_cols = [
    'molecule_chembl_id',
    'canonical_smiles',
    'bioactivity_class',
    'IC50_nM',
    'pIC50',
    'MW',
    'LogP',
    'HBD',
    'HBA',
    'RotB',
    'TPSA',
    'Rings',
    'ArRings'
]

df_final = df_dedup[final_cols].copy()

# Save finalized CSV
df_final.to_csv('keap1_bioactivity_data.csv', index=False)
print(f"Final dataset shape: {df_final.shape}")
df_final.head()

Final dataset shape: (260, 13)


,molecule_chembl_id,canonical_smiles,bioactivity_class,IC50_nM,pIC50,MW,LogP,HBD,HBA,RotB,TPSA,Rings,ArRings
0,CHEMBL2381968,O=C(O)[C@H]1CCCC[C@H]1C(=O)N1CCc2ccccc2[C@H]1C...,intermediate,1068.436299,5.971251,446.50,3.30,1,4,4,94.99,5,2
1,CHEMBL2402205,CC(C)C[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](...,active,33.000000,7.481486,1858.04,-4.81,24,24,61,755.43,3,3
2,CHEMBL2402207,COc1ccc(S(=O)(=O)Nc2ccc(NS(=O)(=O)c3ccc(OC)cc3...,intermediate,1943.767081,5.711356,498.58,4.46,2,6,8,110.80,4,4
3,CHEMBL3134422,CC(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CC(=O)O)C(=O...,active,195.642413,6.708537,1093.15,-3.73,15,15,36,474.42,1,1
4,CHEMBL3235901,CC(C)c1ccc(S(=O)(=O)Nc2cc(Sc3nc[nH]n3)c(O)c3cc...,inactive,100000.000000,4.000000,440.55,4.74,3,6,6,107.97,4,4
